In [0]:
df = spark.table(
    "skillpulse.bronze.bronze_jobs_realtime"
)

print("Bronze Rows:", df.count())

Bronze Rows: 1871


In [0]:
from pyspark.sql.functions import *

df_roles = (
    df
    .withColumn(
        "normalized_role",

        # -------------------------
        # BI Roles
        # -------------------------
        when(
            lower(col("job_title")).rlike(
                "business intelligence|bi analyst|reporting analyst|business systems analyst"
            ),
            "BI Analyst"
        )

        .when(
            lower(col("job_title")).rlike(
                "business intelligence developer|bi developer|business intelligence analyst/developer"
            ),
            "BI Developer"
        )

        # -------------------------
        # Data Science Leadership
        # -------------------------
        .when(
            lower(col("job_title")).rlike(
                "data science manager|head of data science|director data science|senior data science manager"
            ),
            "Data Science Leadership"
        )

        # -------------------------
        # Data Architect
        # -------------------------
        .when(
            lower(col("job_title")).rlike(
                "data solution architect|data solutions architect|data platform architect|data & ai architect|data & ml architect|lead data & ai architect|data & information architect"
            ),
            "Data Architect"
        )

        # -------------------------
        # AI Engineer
        # -------------------------
        .when(
            lower(col("job_title")).rlike(
                "ai|gen ai|genai|generative ai|agentic ai|applied ai|conversational ai"
            ),
            "AI Engineer"
        )

        # -------------------------
        # MLOps Engineer
        # -------------------------
        .when(
            lower(col("job_title")).rlike(
                "mlops|ai devops|ai/ml devops|platform engineer.*ml|platform engineer.*ai"
            ),
            "MLOps Engineer"
        )

        # -------------------------
        # ML Engineer
        # -------------------------
        .when(
            lower(col("job_title")).rlike(
                "machine learning|ml engineer|lead mle|sagemaker|applied scientist|ai/ml consultant"
            ),
            "Machine Learning Engineer"
        )

        # -------------------------
        # Data Engineer
        # -------------------------
        .when(
            lower(col("job_title")).rlike(
                "data reliability engineer|data integration engineer|data developer|engineering lead - data"
            ),
            "Data Engineer"
        )

        .when(
            lower(col("job_title")).rlike(
                "data engineer"
            ),
            "Data Engineer"
        )

        # -------------------------
        # Data Scientist
        # -------------------------
        .when(
            lower(col("job_title")).rlike(
                "data scientist|data science consultant|data science trainee"
            ),
            "Data Scientist"
        )

        # -------------------------
        # Data Analyst
        # -------------------------
        .when(
            lower(col("job_title")).rlike(
                "data analyst|data mapping analyst"
            ),
            "Data Analyst"
        )

        # -------------------------
        # Analytics Engineer
        # -------------------------
        .when(
            lower(col("job_title")).rlike(
                "analytics engineer|analytics platform engineer|analytics solution engineer|analytical engineer"
            ),
            "Analytics Engineer"
        )

        # -------------------------
        # Database Engineer
        # -------------------------
        .when(
            lower(col("job_title")).rlike(
                "database|db engineer|db developer|mongodb engineer|sql developer"
            ),
            "Database Engineer"
        )

        # -------------------------
        # Generic Architect Catch-All
        # -------------------------
        .when(
            lower(col("job_title")).rlike(
                "data architect"
            ),
            "Data Architect"
        )

        .otherwise("Other")
    )
)

In [0]:
(
    df_roles.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "skillpulse.silver.silver_jobs_realtime"
    )
)

print(
    "Silver Jobs Saved:",
    df_roles.count()
)

Silver Jobs Saved: 1871


In [0]:
TECH_SKILLS = [
    "Python",
    "SQL",
    "Spark",
    "PySpark",
    "Databricks",
    "AWS",
    "Azure",
    "GCP",
    "Snowflake",
    "Airflow",
    "Kafka",
    "TensorFlow",
    "PyTorch",
    "Machine Learning",
    "Deep Learning",
    "NLP",
    "Docker",
    "Kubernetes",
    "Terraform",
    "Git",
    "Power BI",
    "Tableau",
    "Data Warehousing",
    "Data Modeling",
    "ETL",
    "Scala",
    "Java",
    "R",
    "NoSQL",
    "Oracle",
    "MySQL",
    "PostgreSQL"
]

SOFT_SKILLS = [
    "Communication",
    "Problem Solving",
    "Teamwork",
    "Leadership",
    "Collaboration",
    "Critical Thinking",
    "Adaptability",
    "Presentation Skills",
    "Time Management",
    "Mentoring"
]

ALL_SKILLS = TECH_SKILLS + SOFT_SKILLS

In [0]:
jobs_df = spark.table(
    "skillpulse.silver.silver_jobs_realtime"
)

In [0]:
from pyspark.sql import Row
import re

rows = []

jobs = jobs_df.select(
    "job_id",
    "job_title",
    "normalized_role",
    "description"
).fillna("").collect()

for job in jobs:

    desc = str(job["description"]).lower()

    for skill in ALL_SKILLS:
        
        pattern = rf"\b{re.escape(skill.lower())}\b"

        if re.search(pattern, desc):
            rows.append(
                Row(
                    job_id=job["job_id"],
                    job_title=job["job_title"],
                    normalized_role=job["normalized_role"],
                    skill=skill
                )
            )

if len(rows) == 0:
    raise Exception("No skills extracted")

skills_df = spark.createDataFrame(rows)

In [0]:
skills_df = skills_df.dropDuplicates(
    ["job_id", "skill"]
)

In [0]:
(
    skills_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "skillpulse.silver.silver_realtime_skills"
    )
)

print(
    "Silver Skills Saved:",
    skills_df.count()
)

Silver Skills Saved: 1067
